# 05 - Anomaly Detection (Isolation Forest)

Estudio experimental para la detección de patrones transaccionales inusuales basados en bosques de aislamiento (Unsupervised Learning).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
import numpy as np
from src.training.train_anomaly_detection import train_isolation_forest, save_anomaly_model
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Simulación de Transacciones Comerciales

In [ ]:
np.random.seed(42)
n_normal = 200
n_anomalous = 10

# Datos normales (Ventas regulares en caja)
normal_sales = pd.DataFrame({
    'discount_pct': np.random.uniform(0.0, 0.1, n_normal),
    'total_amount': np.random.normal(50, 15, n_normal),
    'refund_flag': np.zeros(n_normal)
})

# Anomalías (Anulaciones sospechosas o descuentos autorrecetados)
anomalous_sales = pd.DataFrame({
    'discount_pct': np.random.uniform(0.3, 0.9, n_anomalous),
    'total_amount': np.random.normal(400, 100, n_anomalous),
    'refund_flag': np.ones(n_anomalous)
})

df_auth = pd.concat([normal_sales, anomalous_sales], ignore_index=True)
df_auth = df_auth.sample(frac=1).reset_index(drop=True)
df_auth.head()

### 2. Entrenamiento de Isolation Forest

In [ ]:
# Estimamos un 4.5% de contaminación
clf = train_isolation_forest(df_auth, contamination=0.05)

# Predecir etiquetas anormales (-1)
df_auth['anomaly_score'] = clf.decision_function(df_auth)
df_auth['predicted_tag'] = clf.predict(df_auth)

df_auth['predicted_tag'].value_counts()

### 3. Representación Gráfica

In [ ]:
sns.scatterplot(
    data=df_auth, 
    x='total_amount', 
    y='discount_pct', 
    hue='predicted_tag', 
    palette='coolwarm'
)
plt.title('Detección de Transacciones Anómalas (Riesgo de Fraude)')
plt.show()